# Method: normalisation and shrinkage

*Question → Intuition → Math → Code → Assumptions → How it breaks*

## 1. Question

Ronaldo scored 25 league goals in 2002-03. Haaland scored 36 in 2022-23. Which
is the better season?

The counts are not comparable. They were scored in different leagues, in
different decades, against different defences, under different rules about what
a foul is. Before any ranking can pool them, they have to be put on one scale —
and the choice of scale decides the answer.

## 2. Intuition

Stop asking *how many* and start asking *how far ahead of everyone else*.

A player who scores 25 in a season where the best forwards score 20 has done
something more impressive than one who scores 25 in a season where they score 35.
The comparison that survives across eras is not the raw number, it is the
**distance from the crowd**, measured in units of how spread out the crowd is.

That is a standard score, and it is the whole idea. Everything after it is
detail — but two of those details change the answer, so they get their own
sections below.

## 3. Math

For player $i$ in league-season $g$, with raw rate $x_{ig}$:

$$z_{ig} = \frac{x_{ig} - \mu_g}{\sigma_g}$$

where $\mu_g$ and $\sigma_g$ are the mean and standard deviation of *everyone who
played in that league that season*.

**The denominator is a choice.** We use the population standard deviation
($\text{ddof}=0$), not the sample one ($\text{ddof}=1$). The sample correction
exists to estimate the spread of a larger population from a subset. Here there is
no larger population: the players who played the 2003-04 Premier League *are* the
2003-04 Premier League. Dividing by $n-1$ would be correcting for a sampling step
that never happened.

With $n \approx 500$ per group the numerical difference is tiny. The reason to
get it right is that the alternative is not defensible if someone asks.

In [ ]:
import warnings

import matplotlib
import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.figsize"] = (10, 5.5)

SAMPLE = "../data/sample"
ranking = pd.read_parquet(f"{SAMPLE}/ranking.parquet")
seasons = pd.read_parquet(f"{SAMPLE}/player_season_scored.parquet")
offsets = pd.read_parquet(f"{SAMPLE}/league_offsets.parquet")

Here is the comparison the chapter opened with, done properly. Both
seasons are placed against the players who actually played alongside them.

In [ ]:
from gambeta import level

per90 = seasons[seasons["minutes"] >= 900].copy()
per90["npg_p90"] = per90["npg"] / (per90["minutes"] / 90)

scored = level.zscore(per90, ["npg_p90"])

pair = scored[
    ((scored["player"].str.contains("Ronaldo", na=False)) & (scored["season"] == "0203"))
    | ((scored["player"].str.contains("Haaland", na=False)) & (scored["season"] == "2223"))
]
pair[["player", "season", "league", "npg", "minutes", "npg_p90", "npg_p90_z"]].round(2)

The z-score answers the question the raw count could not. Note it is
computed on non-penalty goals per 90, not total goals: penalties measure who is
assigned them, and per-90 removes the advantage of simply being picked more.

Now the second detail, and it matters more than the first.

## How much of a season's score is signal?

A player with 200 minutes and two goals has a per-90 rate that would lead the
league over a full season. He has not led the league. He has played two hours.

**Reliability** is the word for the share of a measurement that is signal rather
than noise, and it has an operational test: a reliable measurement predicts the
next one, an unreliable measurement does not. So bucket seasons by minutes played
and ask how well each predicts the same player's following season.

In [ ]:
ordered = scored.sort_values(["player_id", "season"]).copy()
ordered["next_z"] = ordered.groupby("player_id")["npg_p90_z"].shift(-1)
consecutive = ordered.dropna(subset=["next_z"])

buckets = pd.cut(
    consecutive["minutes"],
    [900, 1500, 2100, 2700, 10000],
    labels=["900-1500", "1500-2100", "2100-2700", "2700+"],
)
reliability = (
    consecutive.assign(bucket=buckets)
    .groupby("bucket", observed=True)
    .apply(
        lambda g: pd.Series(
            {"seasons": len(g), "predicts next season": g["npg_p90_z"].corr(g["next_z"])}
        ),
        include_groups=False,
    )
    .round(3)
)
print(reliability)
print("\nA short season predicts the next one far worse than a long one does.")
print("That gap is noise, and it is what shrinkage exists to discount.")

## Shrinkage: trusting a season in proportion to its size

The fix is not to discard short seasons — that throws away real players — but to
**pull them toward the average by an amount that depends on how little evidence
they carry**:

$$\tilde{z}_i = z_i \cdot \frac{m_i}{m_i + m_0}$$

where $m_i$ is minutes played and $m_0$ is a prior strength, also in minutes.
At $m_i = m_0$ the score is halved. At $m_i \gg m_0$ almost nothing happens.

This is empirical-Bayes shrinkage written in its simplest possible form. The
weight $m/(m+m_0)$ is what you get from combining a noisy observation with a
prior centred on the mean, when both are treated as normal — the algebra is in
any Bayesian textbook, and the football reading of it is just "believe a season
in proportion to how much of it there was".

`gambeta` uses $m_0 = 900$ minutes: ten full matches.

In [ ]:
example = np.array([2.5, 2.5, 2.5, 2.5])
minutes = np.array([300.0, 900.0, 1800.0, 3200.0])

pd.DataFrame(
    {
        "minutes": minutes.astype(int),
        "raw z": example,
        "shrunk": level.shrink(example, minutes, prior_minutes=900.0).round(2),
        "weight": (minutes / (minutes + 900.0)).round(2),
    }
)

## Is 900 minutes the right prior?

The weight $m/(m+m_0)$ is not an arbitrary curve. It **is** an estimate of
reliability — the share of a measurement that is signal — so it can be checked
against the reliability we just measured, rather than asserted.

If $m_0 = 900$ is roughly right, the weight at each bucket's typical minutes
should track that bucket's observed correlation with the next season.

In [ ]:
typical = consecutive.assign(bucket=buckets).groupby("bucket", observed=True)["minutes"].median()
check = pd.DataFrame(
    {
        "median minutes": typical.astype(int),
        "observed reliability": reliability["predicts next season"],
        "weight at m0=900": (typical / (typical + 900.0)).round(3),
    }
)
print(check.to_string())
print("\nThe two columns move together, which is the most that can be claimed:")
print("m0 = 900 is a defensible order of magnitude, not a fitted value. A")
print("hierarchical model would estimate it, and that is section 3.3 of PENDING.")

Four identical performances, four different amounts of belief. The
900-minute season is halved exactly, which is what the prior means: ten matches
of evidence is worth as much as the prior assumption that you are average.

## 5. Assumptions

1. **A league-season is a closed population.** Nobody outside it is relevant to
   judging a performance inside it. This is what licenses $\text{ddof}=0$, and it
   is also why a player is never compared directly to a different era's raw
   numbers.
2. **The distribution within a group is roughly symmetric.** A z-score is a
   sensible summary when the mean and standard deviation describe the shape. For
   goals per 90 among regulars this is approximately true; for the full squad
   including defenders it is not, which is why the minutes floor exists.
3. **The prior belongs at zero.** Shrinking toward the league average assumes
   that, absent evidence, a player is average. For a randomly chosen player this
   is right by construction.
4. **900 minutes is the right prior strength.** It is not estimated from the
   data. This is the weakest assumption in the chapter and the project says so:
   a hierarchical model would learn it instead.

## 6. How it breaks

Two ways, one visible in the data and one that needs constructing.

**A group too small to have a spread.** The standard deviation of a handful of
players is unstable, and if every player in a group has the same value it is
zero — division by zero. `level.zscore` returns 0 rather than infinity, which is
the safe answer, but it is worth seeing that a "perfectly average" score can mean
"we could not tell".

In [ ]:
tiny = pd.DataFrame(
    {
        "league": ["TEST"] * 3,
        "season": ["0001"] * 3,
        "value": [1.0, 1.0, 1.0],
    }
)
print("three players, identical output, zero spread:")
print(level.zscore(tiny, ["value"])[["value", "value_z"]])
print("\nz = 0 here means 'no information', not 'exactly average ability'.")

**Shrinkage punishes the genuinely brilliant short career.** The
weight does not know *why* a player has few minutes. A 500-minute season from an
injured superstar and a 500-minute season from a squad filler are shrunk by
exactly the same factor.

In [ ]:
short = scored[(scored["minutes"] < 1200) & (scored["npg_p90_z"] > 2.5)]
worst_hit = short.nlargest(8, "npg_p90_z").copy()
worst_hit["shrunk"] = level.shrink(
    worst_hit["npg_p90_z"].to_numpy(), worst_hit["minutes"].to_numpy(dtype=float), 900.0
)
print("real seasons that shrinkage cuts hardest:\n")
print(
    worst_hit[["player", "season", "league", "minutes", "npg_p90_z", "shrunk"]]
    .round(2)
    .to_string(index=False)
)

Every one of those is a real, good half-season being told it counts for
less than it felt like at the time. That is the price of not being fooled by the
hundreds of two-hour hot streaks in the same bucket, and it is a price, not a
free lunch.

The honest summary: **shrinkage trades a specific, known unfairness to individual
short careers for protection against a systematic error affecting the whole
table.** That trade is worth making, and it should be made out loud.